# Applying models

This notebook contains code for evaluating the following models:
- Ordinary Least Squares (OLS) regression
- Stepwise regression
- LASSO regression
- Random forest
- Neural network

*It does not include extensive code on data cleaning (incl. harmonization, encoding, imputation and deletion of various features); the random grid search deployed to derive the neural network's architecture is also not included*

Note that consumption expenditure is currently predicted; a robustness check can be conducted with reference to the "Quintile" column provided by Rwanda's National Institute for Statistics -- this will yield similar results!

In [11]:
# Core libraries
import numpy as np
import pandas as pd

# Scikit-learn models and utilities
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score, recall_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Statistics and scientific computing
from scipy.stats import spearmanr
import statsmodels.api as sm

# Neural network (TensorFlow)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# Progress bar
from tqdm import tqdm

**Import data**

In [12]:
harmonized_features2010_encoded = pd.read_pickle("/Users/gianjaeger/Desktop/Applied Machine Learning Project/harmonized_features2010_encoded.pkl")
harmonized_features2013_encoded = pd.read_pickle("/Users/gianjaeger/Desktop/Applied Machine Learning Project/harmonized_features2013_encoded.pkl")
harmonized_features2016_encoded = pd.read_pickle("/Users/gianjaeger/Desktop/Applied Machine Learning Project/harmonized_features2016_encoded.pkl")

**Prepare for the PMT**

In [13]:
# Remove the HouseholdID column in all dataframes 
for df in [harmonized_features2010_encoded, harmonized_features2013_encoded, harmonized_features2016_encoded]:
    df.drop(columns=['Household ID'], inplace=True)

In [14]:
# Remove the quintile column in all dataframes (the experiment was also conducted with the income quintile value provided as a form of robustness check)
harmonized_features2010_encoded = harmonized_features2010_encoded.drop(columns=['Quintile'])
harmonized_features2013_encoded = harmonized_features2013_encoded.drop(columns=['Quintile'])
harmonized_features2016_encoded = harmonized_features2016_encoded.drop(columns=['Quintile'])

In [15]:
# Log transform consumption
for df in [harmonized_features2010_encoded, harmonized_features2013_encoded, harmonized_features2016_encoded]:
    df['log_consumption'] = np.log(df['Equivalenced consumption'])

In [16]:
# Ensure that all dataframes have the same number of columns (they don't to begin with due to how OneHot Encoding was previously done)

# Build master set of all columns from all three DataFrames
all_columns = set(harmonized_features2010_encoded.columns) | \
              set(harmonized_features2013_encoded.columns) | \
              set(harmonized_features2016_encoded.columns)

def align_columns(df, all_columns):
    for col in all_columns:
        if col not in df.columns:
            df[col] = 0
    return df[list(all_columns)]

# Apply to each DataFrame
harmonized_features2010_encoded = align_columns(harmonized_features2010_encoded, all_columns)
harmonized_features2013_encoded = align_columns(harmonized_features2013_encoded, all_columns)
harmonized_features2016_encoded = align_columns(harmonized_features2016_encoded, all_columns)

**Additional data cleaning incorporated upon final feedback with Emily** 

In [17]:
# Remove the "Total surface area of land owned" column for all dataframes
for df in [harmonized_features2010_encoded, harmonized_features2013_encoded, harmonized_features2016_encoded]:
    df.drop(columns=['Total surface area of land owned'], inplace=True)

In [18]:
# Remove the "Total value of large crop harvest" column for all dataframes
for df in [harmonized_features2010_encoded, harmonized_features2013_encoded, harmonized_features2016_encoded]:
    df.drop(columns=['Total value of large crop harvest'], inplace=True)

In [19]:
# Remove the "Total value of livestock" column for all dataframes
for df in [harmonized_features2010_encoded, harmonized_features2013_encoded, harmonized_features2016_encoded]:
    df.drop(columns=['Total value of livestock'], inplace=True)

In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Define the target columns
columns_to_process = [
    'Distance to the nearest drinking water source (meters)',
    'Amount paid to private water vendor last 7 days'
]

# Function to windsorize and normalize to [0, 1]
def process_dataframe(df):
    df_processed = df.copy()

    for col in columns_to_process:
        # Drop NaNs temporarily
        non_na = df_processed[col].dropna()
        
        # Windsorize: clip at 1st and 99th percentiles
        lower = np.percentile(non_na, 1)
        upper = np.percentile(non_na, 99)
        df_processed[col] = np.clip(df_processed[col], lower, upper)
    
    # Normalize to [0, 1]
    scaler = MinMaxScaler()
    df_processed[columns_to_process] = scaler.fit_transform(df_processed[columns_to_process])

    return df_processed

# Apply to each dataframe
harmonized_features2010_encoded = process_dataframe(harmonized_features2010_encoded)
harmonized_features2013_encoded = process_dataframe(harmonized_features2013_encoded)
harmonized_features2016_encoded = process_dataframe(harmonized_features2016_encoded)

In [21]:
all_columns = set(harmonized_features2010_encoded.columns) | set(harmonized_features2013_encoded.columns) | set(harmonized_features2016_encoded.columns)

def align_columns(df, all_columns):
    for col in all_columns:
        if col not in df.columns:
            df[col] = 0
    return df[list(all_columns)]  # Ensure same column order

# Apply to each dataframe
harmonized_features2010_encoded = align_columns(harmonized_features2010_encoded, all_columns)
harmonized_features2013_encoded = align_columns(harmonized_features2013_encoded, all_columns)
harmonized_features2016_encoded = align_columns(harmonized_features2016_encoded, all_columns)

In [22]:
# Force all columns to numeric
for df in [harmonized_features2010_encoded, harmonized_features2013_encoded, harmonized_features2016_encoded]:
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')

/var/folders/n5/_f2xjmkj39gb7s0th_6lyk9w0000gn/T/ipykernel_7451/900761130.py:4: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


**Defining relevant evaluation metrics**

In [23]:
def exclusion_error_rate(y_true, y_pred, quota):
    cutoff = int(len(y_true) * quota)
    true_poor = set(np.argsort(y_true)[:cutoff])
    pred_poor = set(np.argsort(y_pred)[:cutoff])
    return len(true_poor - pred_poor) / cutoff

def inclusion_error_rate(y_true, y_pred, quota):
    cutoff = int(len(y_true) * quota)
    true_poor = set(np.argsort(y_true)[:cutoff])
    pred_poor = set(np.argsort(y_pred)[:cutoff])
    return len(pred_poor - true_poor) / cutoff

def targeting_error_rate(y_true, y_pred, quota):
    cutoff = int(len(y_true) * quota)
    true_poor = set(np.argsort(y_true)[:cutoff])
    pred_poor = set(np.argsort(y_pred)[:cutoff])
    return len(true_poor.symmetric_difference(pred_poor)) / (2 * cutoff)

---

# OLS

In [24]:
def run_ols_evaluation(train_df, test_df, config_label):
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    
    # Store global metrics separately
    r2_scores = []
    spearman_corrs = []

    # Store threshold-based metrics
    sim_results = {t: {
        'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []
    } for t in thresholds}

    for _ in tqdm(range(100), desc=f"Running {config_label}"):
        if train_df.equals(test_df):
            train_split, test_split = train_test_split(train_df, test_size=0.25)
        else:
            train_split, test_split = train_df.copy(), test_df.copy()

        y_train = train_split['log_consumption']
        y_test = test_split['log_consumption']

        X_train = train_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = test_split.drop(columns=['Equivalenced consumption', 'log_consumption'])

        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Global metrics
        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        # Threshold metrics
        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)

            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    # Global summary row (only one per config)
    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    # Threshold-specific summary
    threshold_summary = []
    for q in thresholds:
        metrics = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(metrics['targeting_error']),
            'ter_std': np.std(metrics['targeting_error']),
            'ee_mean': np.mean(metrics['exclusion_error']),
            'ee_std': np.std(metrics['exclusion_error']),
            'ie_mean': np.mean(metrics['inclusion_error']),
            'ie_std': np.std(metrics['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [25]:
# Run evaluations
global_2010, thresh_2010 = run_ols_evaluation(harmonized_features2010_encoded, harmonized_features2010_encoded, '2010→2010')
global_2013, thresh_2013 = run_ols_evaluation(harmonized_features2010_encoded, harmonized_features2013_encoded, '2010→2013')
global_2016, thresh_2016 = run_ols_evaluation(harmonized_features2010_encoded, harmonized_features2016_encoded, '2010→2016')

# Combine into final DataFrames
global_results_OLS = pd.concat([global_2010, global_2013, global_2016], ignore_index=True)
threshold_results_OLS = pd.concat([thresh_2010, thresh_2013, thresh_2016], ignore_index=True)

Running 2010→2016: 100%|██████████| 100/100 [00:27<00:00,  3.65it/s]


In [26]:
# Run new OLS evaluations (2013 as training set)
global_2013new, thresh_2013new = run_ols_evaluation(harmonized_features2013_encoded, harmonized_features2013_encoded, '2013→2013')
global_1316new, thresh_1316new = run_ols_evaluation(harmonized_features2013_encoded, harmonized_features2016_encoded, '2013→2016')

# Combine into NEW master OLS result frames
global_results_OLS_2013train = pd.concat([global_2013new, global_1316new], ignore_index=True)
threshold_results_OLS_2013train = pd.concat([thresh_2013new, thresh_1316new], ignore_index=True)

Running 2013→2016: 100%|██████████| 100/100 [00:18<00:00,  5.31it/s]


In [28]:
# Combine results across all 5 configs
global_results_OLS_all = pd.concat([global_results_OLS, global_results_OLS_2013train],ignore_index=True)
threshold_results_OLS_all = pd.concat([threshold_results_OLS, threshold_results_OLS_2013train],ignore_index=True)

# Save results to CSV
global_results_OLS_all.to_csv("global_results_OLS_all.csv", index=False)
threshold_results_OLS_all.to_csv("threshold_results_OLS_all.csv", index=False)

# Domain classifier (mitigating data decay)

In [ ]:
# Relevant for all models

def create_domain_dataset(X_source, X_target):
    X_domain = pd.concat([X_source, X_target], ignore_index=True)
    y_domain = np.concatenate([np.zeros(len(X_source)), np.ones(len(X_target))])
    return X_domain, y_domain

def train_domain_classifier(X_domain, y_domain):
    clf = LogisticRegression(solver='lbfgs', max_iter=1000)
    clf.fit(X_domain, y_domain)
    return clf

def compute_importance_weights(clf, X_source, epsilon=0.01):
    D_x = clf.predict_proba(X_source)[:, 1]  # P(d=1 | x)
    D_clipped = np.clip(D_x, epsilon, 1 - epsilon)
    beta = D_clipped / (1 - D_clipped)
    return beta / np.mean(beta)

**OLS mitgated for data deday with importance weights**

In [34]:
def run_ols_evaluation(train_df, test_df, config_label, weights=None):
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    r2_scores, spearman_corrs = [], []
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for i in tqdm(range(100), desc=f"Running {config_label}"):
        if train_df.equals(test_df):
            train_split, test_split = train_test_split(train_df, test_size=0.25)
            weight_split = weights.loc[train_split.index] if weights is not None else None
        else:
            train_split, test_split = train_df.copy(), test_df.copy()
            weight_split = weights if weights is not None else None

        y_train = train_split['log_consumption']
        y_test = test_split['log_consumption']
        X_train = train_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = test_split.drop(columns=['Equivalenced consumption', 'log_consumption'])

        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        # Model training
        model = LinearRegression()
        model.fit(X_train, y_train, sample_weight=weight_split)

        y_pred = model.predict(X_test)

        # Metrics
        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)
            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    threshold_summary = []
    for q in thresholds:
        m = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(m['targeting_error']),
            'ter_std': np.std(m['targeting_error']),
            'ee_mean': np.mean(m['exclusion_error']),
            'ee_std': np.std(m['exclusion_error']),
            'ie_mean': np.mean(m['inclusion_error']),
            'ie_std': np.std(m['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [36]:
# Step 1: Compute domain weights for each test set

# 2010 → 2013 weights
X_domain_1030, y_domain_1030 = create_domain_dataset(
    harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1030 = train_domain_classifier(X_domain_1030, y_domain_1030)
weights_1030 = compute_importance_weights(clf_1030, harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1030 = pd.Series(weights_1030, index=harmonized_features2010_encoded.index)

# 2010 → 2016 weights
X_domain_1060, y_domain_1060 = create_domain_dataset(
    harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2016_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1060 = train_domain_classifier(X_domain_1060, y_domain_1060)
weights_1060 = compute_importance_weights(clf_1060, harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1060 = pd.Series(weights_1060, index=harmonized_features2010_encoded.index)

# 2013 → 2016 weights
X_domain_1360, y_domain_1360 = create_domain_dataset(
    harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2016_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1360 = train_domain_classifier(X_domain_1360, y_domain_1360)
weights_1360 = compute_importance_weights(clf_1360, harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1360 = pd.Series(weights_1360, index=harmonized_features2013_encoded.index)

# Step 2: Run weighted evaluations using these weights
global_2013_w, thresh_2013_w = run_ols_evaluation(harmonized_features2010_encoded, harmonized_features2013_encoded, '2010→2013 (IW)', weights_1030)
global_2016_w, thresh_2016_w = run_ols_evaluation(harmonized_features2010_encoded, harmonized_features2016_encoded, '2010→2016 (IW)', weights_1060)
global_1360_w, thresh_1360_w = run_ols_evaluation(harmonized_features2013_encoded, harmonized_features2016_encoded, '2013→2016 (IW)', weights_1360)

# Step 3: Combine all IW results across configurations
global_results_IW_all = pd.concat([global_2013_w, global_2016_w, global_1360_w], ignore_index=True)
threshold_results_IW_all = pd.concat([thresh_2013_w, thresh_2016_w, thresh_1360_w], ignore_index=True)

# Step 4: Save to CSV
global_results_IW_all.to_csv("global_results_IW_all.csv", index=False)
threshold_results_IW_all.to_csv("threshold_results_IW_all.csv", index=False)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

# Stepwise Regression

In [37]:
def run_stepwise_evaluation(train_df, test_df, config_label, max_features=15, n_simulations=10):
    from sklearn.metrics import mean_squared_error

    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    r2_scores = []
    spearman_corrs = []
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for _ in tqdm(range(n_simulations), desc=f"Running {config_label} [Stepwise]"):
        if train_df.equals(test_df):
            full_train, test_split = train_test_split(train_df, test_size=0.25)
        else:
            full_train, test_split = train_df.copy(), test_df.copy()

        # Selection & calibration split
        selection_set, calibration_set = train_test_split(full_train, test_size=0.25)

        # Pre-screen features:
        all_features = selection_set.drop(columns=['Equivalenced consumption', 'log_consumption'])
        
        # Drop constant features (zero variance)
        non_constant = all_features.loc[:, all_features.nunique() > 1]

        # Filter by Pearson correlation with the target
        correlations = non_constant.apply(lambda col: np.corrcoef(col, selection_set['log_consumption'])[0, 1])
        screened_features = correlations[correlations.abs() > 0.01].dropna().index.tolist()

        selected_features = []
        best_mse = np.inf

        y_selection = selection_set['log_consumption']
        y_calibration = calibration_set['log_consumption']

        for i in range(max_features):
            remaining = list(set(screened_features) - set(selected_features))
            best_candidate = None
            best_candidate_mse = best_mse

            for feature in remaining:
                trial_features = selected_features + [feature]

                X_sel = selection_set[trial_features]
                X_cal = calibration_set[trial_features]

                model = LinearRegression()
                model.fit(X_sel, y_selection)
                y_cal_pred = model.predict(X_cal)
                mse = mean_squared_error(y_calibration, y_cal_pred)

                if mse < best_candidate_mse:
                    best_candidate_mse = mse
                    best_candidate = feature

            if best_candidate is not None and best_candidate_mse < best_mse:
                selected_features.append(best_candidate)
                best_mse = best_candidate_mse
            else:
                break  # No improvement → stop

        # Retrain final model on full training set with selected features
        y_train = full_train['log_consumption']
        y_test = test_split['log_consumption']

        X_train = full_train[selected_features]
        X_test = test_split[selected_features].reindex(columns=X_train.columns, fill_value=0)

        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Global metrics
        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)

            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    # Global summary
    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    # Threshold-level summary
    threshold_summary = []
    for q in thresholds:
        metrics = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(metrics['targeting_error']),
            'ter_std': np.std(metrics['targeting_error']),
            'ee_mean': np.mean(metrics['exclusion_error']),
            'ee_std': np.std(metrics['exclusion_error']),
            'ie_mean': np.mean(metrics['inclusion_error']),
            'ie_std': np.std(metrics['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [38]:
# Run Stepwise regression evaluations
global_2010_s, thresh_2010_s = run_stepwise_evaluation(harmonized_features2010_encoded, harmonized_features2010_encoded, '2010→2010')
global_2013_s, thresh_2013_s = run_stepwise_evaluation(harmonized_features2010_encoded, harmonized_features2013_encoded, '2010→2013')
global_2016_s, thresh_2016_s = run_stepwise_evaluation(harmonized_features2010_encoded, harmonized_features2016_encoded, '2010→2016')

# Combine into final summary DataFrames
global_results_stepwise = pd.concat([global_2010_s, global_2013_s, global_2016_s], ignore_index=True)
threshold_results_stepwise = pd.concat([thresh_2010_s, thresh_2013_s, thresh_2016_s], ignore_index=True)

Running 2010→2010 [Stepwise]:   0%|          | 0/10 [00:00<?, ?it/s]

Running 2010→2016 [Stepwise]: 100%|██████████| 10/10 [02:13<00:00, 13.34s/it]


In [39]:
# Run new Stepwise evaluations (2013 as training set)
global_2013new_s, thresh_2013new_s = run_stepwise_evaluation(harmonized_features2013_encoded, harmonized_features2013_encoded, '2013→2013')
global_1316new_s, thresh_1316new_s = run_stepwise_evaluation(harmonized_features2013_encoded, harmonized_features2016_encoded, '2013→2016')

# Combine into NEW stepwise results
global_results_stepwise_2013train = pd.concat([global_2013new_s, global_1316new_s], ignore_index=True)
threshold_results_stepwise_2013train = pd.concat([thresh_2013new_s, thresh_1316new_s], ignore_index=True)

Running 2013→2016 [Stepwise]: 100%|██████████| 10/10 [01:08<00:00,  6.83s/it]


In [40]:
# Combine all 5 stepwise configs into full summary tables
global_results_stepwise_all = pd.concat([global_results_stepwise, global_results_stepwise_2013train],ignore_index=True)
threshold_results_stepwise_all = pd.concat([threshold_results_stepwise, threshold_results_stepwise_2013train],ignore_index=True)

# Save to CSV
global_results_stepwise_all.to_csv("global_results_stepwise_all.csv", index=False)
threshold_results_stepwise_all.to_csv("threshold_results_stepwise_all.csv", index=False)

**Stepwise mitgated for data deday with importance weights**

In [42]:
def run_stepwise_evaluation(train_df, test_df, config_label, max_features=15, n_simulations=10, weights=None):
    from sklearn.metrics import mean_squared_error

    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    r2_scores = []
    spearman_corrs = []
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for _ in tqdm(range(n_simulations), desc=f"Running {config_label} [Stepwise]"):
        if train_df.equals(test_df):
            full_train, test_split = train_test_split(train_df, test_size=0.25)
            weight_split = weights.loc[full_train.index] if weights is not None else None
        else:
            full_train, test_split = train_df.copy(), test_df.copy()
            weight_split = weights if weights is not None else None

        # Selection & calibration split
        selection_set, calibration_set = train_test_split(full_train, test_size=0.25)

        # Pre-screen features:
        all_features = selection_set.drop(columns=['Equivalenced consumption', 'log_consumption'])
        
        # Drop constant features (zero variance)
        non_constant = all_features.loc[:, all_features.nunique() > 1]

        # Filter by Pearson correlation with the target
        correlations = non_constant.apply(lambda col: np.corrcoef(col, selection_set['log_consumption'])[0, 1])
        screened_features = correlations[correlations.abs() > 0.01].dropna().index.tolist()

        selected_features = []
        best_mse = np.inf

        y_selection = selection_set['log_consumption']
        y_calibration = calibration_set['log_consumption']

        for _ in range(max_features):
            remaining = list(set(screened_features) - set(selected_features))
            best_candidate = None
            best_candidate_mse = best_mse

            for feature in remaining:
                trial_features = selected_features + [feature]

                X_sel = selection_set[trial_features]
                X_cal = calibration_set[trial_features]

                model = LinearRegression()
                model.fit(X_sel, y_selection)
                y_cal_pred = model.predict(X_cal)
                mse = mean_squared_error(y_calibration, y_cal_pred)

                if mse < best_candidate_mse:
                    best_candidate_mse = mse
                    best_candidate = feature

            if best_candidate is not None and best_candidate_mse < best_mse:
                selected_features.append(best_candidate)
                best_mse = best_candidate_mse
            else:
                break  # No improvement → stop

        # Final model training
        y_train = full_train['log_consumption']
        y_test = test_split['log_consumption']

        X_train = full_train[selected_features]
        X_test = test_split[selected_features].reindex(columns=X_train.columns, fill_value=0)

        model = LinearRegression()
        if weight_split is not None:
            model.fit(X_train, y_train, sample_weight=weight_split)
        else:
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        # Global metrics
        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)

            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    # Global summary
    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    # Threshold-level summary
    threshold_summary = []
    for q in thresholds:
        metrics = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(metrics['targeting_error']),
            'ter_std': np.std(metrics['targeting_error']),
            'ee_mean': np.mean(metrics['exclusion_error']),
            'ee_std': np.std(metrics['exclusion_error']),
            'ie_mean': np.mean(metrics['inclusion_error']),
            'ie_std': np.std(metrics['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [44]:
# Step 1: Compute domain weights for each test set

# 2010 → 2013 weights
X_domain_1030, y_domain_1030 = create_domain_dataset(
    harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1030 = train_domain_classifier(X_domain_1030, y_domain_1030)
weights_1030 = compute_importance_weights(clf_1030, harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1030 = pd.Series(weights_1030, index=harmonized_features2010_encoded.index)

# 2010 → 2016 weights
X_domain_1060, y_domain_1060 = create_domain_dataset(
    harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2016_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1060 = train_domain_classifier(X_domain_1060, y_domain_1060)
weights_1060 = compute_importance_weights(clf_1060, harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1060 = pd.Series(weights_1060, index=harmonized_features2010_encoded.index)

# 2013 → 2016 weights
X_domain_1360, y_domain_1360 = create_domain_dataset(
    harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2016_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1360 = train_domain_classifier(X_domain_1360, y_domain_1360)
weights_1360 = compute_importance_weights(clf_1360, harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1360 = pd.Series(weights_1360, index=harmonized_features2013_encoded.index)

# Step 2: Run importance-weighted stepwise regression
global_sw_2013, thresh_sw_2013 = run_stepwise_evaluation(
    harmonized_features2010_encoded, harmonized_features2013_encoded, '2010→2013 (IW Stepwise)', n_simulations=10, weights=weights_1030
)

global_sw_2016, thresh_sw_2016 = run_stepwise_evaluation(
    harmonized_features2010_encoded, harmonized_features2016_encoded, '2010→2016 (IW Stepwise)', n_simulations=10, weights=weights_1060
)

global_sw_1360, thresh_sw_1360 = run_stepwise_evaluation(
    harmonized_features2013_encoded, harmonized_features2016_encoded, '2013→2016 (IW Stepwise)', n_simulations=10, weights=weights_1360
)

# Step 3: Combine all stepwise IW results across configurations
global_results_IW_stepwise_all = pd.concat([global_sw_2013, global_sw_2016, global_sw_1360], ignore_index=True)
threshold_results_IW_stepwise_all = pd.concat([thresh_sw_2013, thresh_sw_2016, thresh_sw_1360], ignore_index=True)

# Step 4: Save to CSV
global_results_IW_stepwise_all.to_csv("global_results_IW_stepwise_all.csv", index=False)
threshold_results_IW_stepwise_all.to_csv("threshold_results_IW_stepwise_all.csv", index=False)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

# LASSO

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

def run_lasso_evaluation(train_df, test_df, config_label, n_simulations=100):
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    r2_scores = []
    spearman_corrs = []
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for _ in tqdm(range(n_simulations), desc=f"Running {config_label} [Lasso]"):
        if train_df.equals(test_df):
            train_split, test_split = train_test_split(train_df, test_size=0.25)
        else:
            train_split, test_split = train_df.copy(), test_df.copy()

        y_train = train_split['log_consumption']
        y_test = test_split['log_consumption']

        X_train = train_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = test_split.drop(columns=['Equivalenced consumption', 'log_consumption'])

        # Align test columns
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        # Standardize features (important for Lasso)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Lasso with 3-fold CV (on training set only)
        lasso_cv = LassoCV(cv=3, random_state=42, n_jobs=-1)
        lasso_cv.fit(X_train_scaled, y_train)

        y_pred = lasso_cv.predict(X_test_scaled)

        # Global metrics
        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)

            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    # Global summary
    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    # Threshold-based summary
    threshold_summary = []
    for q in thresholds:
        metrics = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(metrics['targeting_error']),
            'ter_std': np.std(metrics['targeting_error']),
            'ee_mean': np.mean(metrics['exclusion_error']),
            'ee_std': np.std(metrics['exclusion_error']),
            'ie_mean': np.mean(metrics['inclusion_error']),
            'ie_std': np.std(metrics['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [46]:
global_2010_l, thresh_2010_l = run_lasso_evaluation(harmonized_features2010_encoded, harmonized_features2010_encoded, '2010→2010')
global_2013_l, thresh_2013_l = run_lasso_evaluation(harmonized_features2010_encoded, harmonized_features2013_encoded, '2010→2013')
global_2016_l, thresh_2016_l = run_lasso_evaluation(harmonized_features2010_encoded, harmonized_features2016_encoded, '2010→2016')

# Combine into tidy result tables
global_results_lasso = pd.concat([global_2010_l, global_2013_l, global_2016_l], ignore_index=True)
threshold_results_lasso = pd.concat([thresh_2010_l, thresh_2013_l, thresh_2016_l], ignore_index=True)

Running 2010→2016 [Lasso]: 100%|██████████| 10/10 [00:11<00:00,  1.16s/it]


In [47]:
# Run new Lasso evaluations (2013 as training set)
global_2013new_l, thresh_2013new_l = run_lasso_evaluation(harmonized_features2013_encoded, harmonized_features2013_encoded, '2013→2013')
global_1316new_l, thresh_1316new_l = run_lasso_evaluation(harmonized_features2013_encoded, harmonized_features2016_encoded, '2013→2016')

# Combine into NEW lasso results
global_results_lasso_2013train = pd.concat([global_2013new_l, global_1316new_l], ignore_index=True)
threshold_results_lasso_2013train = pd.concat([thresh_2013new_l, thresh_1316new_l], ignore_index=True)

Running 2013→2016 [Lasso]:   0%|          | 0/10 [00:00<?, ?it/s]/var/folders/n5/_f2xjmkj39gb7s0th_6lyk9w0000gn/T/ipykernel_7451/554041463.py:38: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corrs.append(spearmanr(y_test, y_pred).correlation)
Running 2013→2016 [Lasso]:  10%|█         | 1/10 [00:00<00:04,  2.23it/s]/var/folders/n5/_f2xjmkj39gb7s0th_6lyk9w0000gn/T/ipykernel_7451/554041463.py:38: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corrs.append(spearmanr(y_test, y_pred).correlation)
Running 2013→2016 [Lasso]:  20%|██        | 2/10 [00:02<00:09,  1.14s/it]/var/folders/n5/_f2xjmkj39gb7s0th_6lyk9w0000gn/T/ipykernel_7451/554041463.py:38: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corrs.append(spearmanr(y_test, y_pred).correlation)
Running 2013→2016 [Lasso]:  30%|███       | 3/10 [00:02<00:05,  1.26it/s]/v

In [48]:
# Combine all 5 lasso configs into full summary tables
global_results_lasso_all = pd.concat([global_results_lasso, global_results_lasso_2013train],ignore_index=True)
threshold_results_lasso_all = pd.concat([threshold_results_lasso, threshold_results_lasso_2013train],ignore_index=True)

# Save to CSV
global_results_lasso_all.to_csv("global_results_lasso_all.csv", index=False)
threshold_results_lasso_all.to_csv("threshold_results_lasso_all.csv", index=False)

**Stepwise mitgated for data deday with importance weights**

In [49]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

def run_lasso_evaluation(train_df, test_df, config_label, n_simulations=100, weights=None):
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    r2_scores = []
    spearman_corrs = []
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for _ in tqdm(range(n_simulations), desc=f"Running {config_label} [Lasso]"):
        if train_df.equals(test_df):
            train_split, test_split = train_test_split(train_df, test_size=0.25)
            weight_split = weights.loc[train_split.index] if weights is not None else None
        else:
            train_split, test_split = train_df.copy(), test_df.copy()
            weight_split = weights if weights is not None else None

        y_train = train_split['log_consumption']
        y_test = test_split['log_consumption']

        X_train = train_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = test_split.drop(columns=['Equivalenced consumption', 'log_consumption'])

        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        # Standardize
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Fit Lasso with sample weights if provided
        lasso_cv = LassoCV(cv=3, random_state=42, n_jobs=-1)
        lasso_cv.fit(X_train_scaled, y_train, sample_weight=weight_split)

        y_pred = lasso_cv.predict(X_test_scaled)

        # Global metrics
        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)

            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    threshold_summary = []
    for q in thresholds:
        metrics = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(metrics['targeting_error']),
            'ter_std': np.std(metrics['targeting_error']),
            'ee_mean': np.mean(metrics['exclusion_error']),
            'ee_std': np.std(metrics['exclusion_error']),
            'ie_mean': np.mean(metrics['inclusion_error']),
            'ie_std': np.std(metrics['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [50]:
# Compute domain weights as before

# 2010 → 2013 weights
X_domain_1030, y_domain_1030 = create_domain_dataset(
    harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1030 = train_domain_classifier(X_domain_1030, y_domain_1030)
weights_1030 = compute_importance_weights(clf_1030, harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1030 = pd.Series(weights_1030, index=harmonized_features2010_encoded.index)

# 2010 → 2016 weights
X_domain_1060, y_domain_1060 = create_domain_dataset(
    harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2016_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1060 = train_domain_classifier(X_domain_1060, y_domain_1060)
weights_1060 = compute_importance_weights(clf_1060, harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1060 = pd.Series(weights_1060, index=harmonized_features2010_encoded.index)

# 2013 → 2016 weights
X_domain_1360, y_domain_1360 = create_domain_dataset(
    harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2016_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1360 = train_domain_classifier(X_domain_1360, y_domain_1360)
weights_1360 = compute_importance_weights(clf_1360, harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1360 = pd.Series(weights_1360, index=harmonized_features2013_encoded.index)

# Run Lasso IW evaluations

global_lasso_2013, thresh_lasso_2013 = run_lasso_evaluation(
    harmonized_features2010_encoded, harmonized_features2013_encoded, '2010→2013 (IW Lasso)', weights=weights_1030
)

global_lasso_2016, thresh_lasso_2016 = run_lasso_evaluation(
    harmonized_features2010_encoded, harmonized_features2016_encoded, '2010→2016 (IW Lasso)', weights=weights_1060
)

global_lasso_1360, thresh_lasso_1360 = run_lasso_evaluation(
    harmonized_features2013_encoded, harmonized_features2016_encoded, '2013→2016 (IW Lasso)', weights=weights_1360
)

# Combine and save

global_results_IW_lasso_all = pd.concat([global_lasso_2013, global_lasso_2016, global_lasso_1360], ignore_index=True)
threshold_results_IW_lasso_all = pd.concat([thresh_lasso_2013, thresh_lasso_2016, thresh_lasso_1360], ignore_index=True)

global_results_IW_lasso_all.to_csv("global_results_IW_lasso_all.csv", index=False)
threshold_results_IW_lasso_all.to_csv("threshold_results_IW_lasso_all.csv", index=False)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

# Random Forest

In [66]:
def run_rf_evaluation(train_df, test_df, config_label, rf_params, n_simulations=5):
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    r2_scores = []
    spearman_corrs = []
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for _ in tqdm(range(n_simulations), desc=f"Running {config_label} [RF fixed params]"):
        if train_df.equals(test_df):
            train_split, test_split = train_test_split(train_df, test_size=0.25)
        else:
            train_split, test_split = train_df.copy(), test_df.copy()

        y_train = train_split['log_consumption']
        y_test = test_split['log_consumption']

        X_train = train_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = test_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        model = RandomForestRegressor(random_state=42, **rf_params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)
            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    threshold_summary = []
    for q in thresholds:
        m = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(m['targeting_error']),
            'ter_std': np.std(m['targeting_error']),
            'ee_mean': np.mean(m['exclusion_error']),
            'ee_std': np.std(m['exclusion_error']),
            'ie_mean': np.mean(m['inclusion_error']),
            'ie_std': np.std(m['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [68]:
def find_best_rf_params(train_df):
    X = train_df.drop(columns=['Equivalenced consumption', 'log_consumption'])
    y = train_df['log_consumption']

    param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    rf_model = RandomForestRegressor(random_state=42)

    rf_cv = RandomizedSearchCV(
        rf_model,
        param_distributions=param_dist,
        n_iter=10,
        cv=3,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )

    rf_cv.fit(X, y)
    print("Best parameters found:", rf_cv.best_params_)  # <- Added print
    return rf_cv.best_params_

# Tune for 2010 and 2013 training sets
rf_params_2010 = find_best_rf_params(harmonized_features2010_encoded)
rf_params_2013 = find_best_rf_params(harmonized_features2013_encoded)

Best parameters found: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 10}
Best parameters found: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 5}


In [69]:
# 2010-trained RF evaluations
global_2010_rf, thresh_2010_rf = run_rf_evaluation(
    harmonized_features2010_encoded, harmonized_features2010_encoded, '2010→2010', rf_params=rf_params_2010)

global_2013_rf, thresh_2013_rf = run_rf_evaluation(
    harmonized_features2010_encoded, harmonized_features2013_encoded, '2010→2013', rf_params=rf_params_2010)

global_2016_rf, thresh_2016_rf = run_rf_evaluation(
    harmonized_features2010_encoded, harmonized_features2016_encoded, '2010→2016', rf_params=rf_params_2010)

# Combine 2010-based results
global_results_rf = pd.concat([global_2010_rf, global_2013_rf, global_2016_rf], ignore_index=True)
threshold_results_rf = pd.concat([thresh_2010_rf, thresh_2013_rf, thresh_2016_rf], ignore_index=True)

# 2013-trained RF evaluations
global_2013new_rf, thresh_2013new_rf = run_rf_evaluation(
    harmonized_features2013_encoded, harmonized_features2013_encoded, '2013→2013', rf_params=rf_params_2013)

global_1316new_rf, thresh_1316new_rf = run_rf_evaluation(
    harmonized_features2013_encoded, harmonized_features2016_encoded, '2013→2016', rf_params=rf_params_2013)

# Combine 2013-based results
global_results_rf_2013train = pd.concat([global_2013new_rf, global_1316new_rf], ignore_index=True)
threshold_results_rf_2013train = pd.concat([thresh_2013new_rf, thresh_1316new_rf], ignore_index=True)

Running 2013→2016 [RF fixed params]: 100%|██████████| 5/5 [00:08<00:00,  1.68s/it]


In [70]:
# Combine all RF configs into full summary tables
global_results_rf_all = pd.concat([global_results_rf, global_results_rf_2013train], ignore_index=True)
threshold_results_rf_all = pd.concat([threshold_results_rf, threshold_results_rf_2013train], ignore_index=True)

# Save to CSV
global_results_rf_all.to_csv("global_results_rf_all.csv", index=False)
threshold_results_rf_all.to_csv("threshold_results_rf_all.csv", index=False)

**Random forest mitgated for data deday with importance weights**

In [72]:
def run_rf_evaluation(train_df, test_df, config_label, rf_params, n_simulations=5, weights=None):
    r2_scores = []
    spearman_corrs = []
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for _ in tqdm(range(n_simulations), desc=f"Running {config_label} [RF fixed params]"):
        if train_df.equals(test_df):
            train_split, test_split = train_test_split(train_df, test_size=0.25)
            weight_split = weights.loc[train_split.index] if weights is not None else None
        else:
            train_split, test_split = train_df.copy(), test_df.copy()
            weight_split = weights if weights is not None else None

        y_train = train_split['log_consumption']
        y_test = test_split['log_consumption']

        X_train = train_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = test_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        model = RandomForestRegressor(random_state=42, **rf_params)
        model.fit(X_train, y_train, sample_weight=weight_split)
        y_pred = model.predict(X_test)

        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)
            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    threshold_summary = []
    for q in thresholds:
        m = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(m['targeting_error']),
            'ter_std': np.std(m['targeting_error']),
            'ee_mean': np.mean(m['exclusion_error']),
            'ee_std': np.std(m['exclusion_error']),
            'ie_mean': np.mean(m['inclusion_error']),
            'ie_std': np.std(m['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [73]:
# Compute domain weights

# 2010 → 2013
X_domain_1030, y_domain_1030 = create_domain_dataset(
    harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1030 = train_domain_classifier(X_domain_1030, y_domain_1030)
weights_1030 = compute_importance_weights(clf_1030, harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1030 = pd.Series(weights_1030, index=harmonized_features2010_encoded.index)

# 2010 → 2016
X_domain_1060, y_domain_1060 = create_domain_dataset(
    harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2016_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1060 = train_domain_classifier(X_domain_1060, y_domain_1060)
weights_1060 = compute_importance_weights(clf_1060, harmonized_features2010_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1060 = pd.Series(weights_1060, index=harmonized_features2010_encoded.index)

# 2013 → 2016
X_domain_1360, y_domain_1360 = create_domain_dataset(
    harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']),
    harmonized_features2016_encoded.drop(columns=['Equivalenced consumption', 'log_consumption'])
)
clf_1360 = train_domain_classifier(X_domain_1360, y_domain_1360)
weights_1360 = compute_importance_weights(clf_1360, harmonized_features2013_encoded.drop(columns=['Equivalenced consumption', 'log_consumption']))
weights_1360 = pd.Series(weights_1360, index=harmonized_features2013_encoded.index)

# Define RF hyperparameters
best_rf_params_2010 = {
    'n_estimators': 200,
    'max_depth': 10,
    'min_samples_split': 2,
    'min_samples_leaf': 2
}

best_rf_params_2013 = {
    'n_estimators': 200,
    'max_depth': 5,
    'min_samples_split': 5,
    'min_samples_leaf': 1
}

# Run IW RF evaluations using the appropriate parameter set
global_rf_2013, thresh_rf_2013 = run_rf_evaluation(
    harmonized_features2010_encoded, harmonized_features2013_encoded,
    '2010→2013 (IW RF)', rf_params=best_rf_params_2010, weights=weights_1030
)

global_rf_2016, thresh_rf_2016 = run_rf_evaluation(
    harmonized_features2010_encoded, harmonized_features2016_encoded,
    '2010→2016 (IW RF)', rf_params=best_rf_params_2010, weights=weights_1060
)

global_rf_1360, thresh_rf_1360 = run_rf_evaluation(
    harmonized_features2013_encoded, harmonized_features2016_encoded,
    '2013→2016 (IW RF)', rf_params=best_rf_params_2013, weights=weights_1360
)

# Combine and save
global_results_IW_rf_all = pd.concat([global_rf_2013, global_rf_2016, global_rf_1360], ignore_index=True)
threshold_results_IW_rf_all = pd.concat([thresh_rf_2013, thresh_rf_2016, thresh_rf_1360], ignore_index=True)

global_results_IW_rf_all.to_csv("global_results_IW_rf_all.csv", index=False)
threshold_results_IW_rf_all.to_csv("threshold_results_IW_rf_all.csv", index=False)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

# Gradient Boosting machine

In [74]:
from sklearn.ensemble import GradientBoostingRegressor

def run_gbm_evaluation(train_df, test_df, config_label, gbm_params, n_simulations=5):
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    r2_scores = []
    spearman_corrs = []
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for _ in tqdm(range(n_simulations), desc=f"Running {config_label} [GBM fixed params]"):
        if train_df.equals(test_df):
            train_split, test_split = train_test_split(train_df, test_size=0.25)
        else:
            train_split, test_split = train_df.copy(), test_df.copy()

        y_train = train_split['log_consumption']
        y_test = test_split['log_consumption']

        X_train = train_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = test_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        model = GradientBoostingRegressor(random_state=42, **gbm_params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)
            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    threshold_summary = []
    for q in thresholds:
        m = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(m['targeting_error']),
            'ter_std': np.std(m['targeting_error']),
            'ee_mean': np.mean(m['exclusion_error']),
            'ee_std': np.std(m['exclusion_error']),
            'ie_mean': np.mean(m['inclusion_error']),
            'ie_std': np.std(m['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [75]:
def find_best_gbm_params(train_df):
    X = train_df.drop(columns=['Equivalenced consumption', 'log_consumption'])
    y = train_df['log_consumption']

    param_dist = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 10],
        'min_samples_leaf': [1, 5, 10],
        'learning_rate': [0.01, 0.05, 0.1]
    }

    gbm_model = GradientBoostingRegressor(random_state=42)

    gbm_cv = RandomizedSearchCV(
        gbm_model,
        param_distributions=param_dist,
        n_iter=10,
        cv=3,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )

    gbm_cv.fit(X, y)
    print("Best GBM parameters found:", gbm_cv.best_params_)
    return gbm_cv.best_params_

In [77]:
gbm_params_2010 = find_best_gbm_params(harmonized_features2010_encoded)
print("Chosen GBM hyperparameters for 2010:", gbm_params_2010)

gbm_params_2013 = find_best_gbm_params(harmonized_features2013_encoded)
print("Chosen GBM hyperparameters for 2013:", gbm_params_2013)

Best GBM parameters found: {'n_estimators': 200, 'min_samples_leaf': 1, 'max_depth': 3, 'learning_rate': 0.05}
Chosen GBM hyperparameters for 2010: {'n_estimators': 200, 'min_samples_leaf': 1, 'max_depth': 3, 'learning_rate': 0.05}
Best GBM parameters found: {'n_estimators': 200, 'min_samples_leaf': 10, 'max_depth': 3, 'learning_rate': 0.01}
Chosen GBM hyperparameters for 2013: {'n_estimators': 200, 'min_samples_leaf': 10, 'max_depth': 3, 'learning_rate': 0.01}


In [78]:
# 2010-trained GBM evaluations
global_2010_gbm, thresh_2010_gbm = run_gbm_evaluation(
    harmonized_features2010_encoded, harmonized_features2010_encoded, '2010→2010 [GBM]', gbm_params=gbm_params_2010)

global_2013_gbm, thresh_2013_gbm = run_gbm_evaluation(
    harmonized_features2010_encoded, harmonized_features2013_encoded, '2010→2013 [GBM]', gbm_params=gbm_params_2010)

global_2016_gbm, thresh_2016_gbm = run_gbm_evaluation(
    harmonized_features2010_encoded, harmonized_features2016_encoded, '2010→2016 [GBM]', gbm_params=gbm_params_2010)

# 2013-trained GBM evaluations
global_2013new_gbm, thresh_2013new_gbm = run_gbm_evaluation(
    harmonized_features2013_encoded, harmonized_features2013_encoded, '2013→2013 [GBM]', gbm_params=gbm_params_2013)

global_1316new_gbm, thresh_1316new_gbm = run_gbm_evaluation(
    harmonized_features2013_encoded, harmonized_features2016_encoded, '2013→2016 [GBM]', gbm_params=gbm_params_2013)

Running 2013→2016 [GBM] [GBM fixed params]: 100%|██████████| 5/5 [00:08<00:00,  1.76s/it]


In [ ]:
# Combine all GBM results
global_results_gbm_all = pd.concat([
    global_2010_gbm, global_2013_gbm, global_2016_gbm,
    global_2013new_gbm, global_1316new_gbm
], ignore_index=True)

threshold_results_gbm_all = pd.concat([
    thresh_2010_gbm, thresh_2013_gbm, thresh_2016_gbm,
    thresh_2013new_gbm, thresh_1316new_gbm
], ignore_index=True)

# Save to CSV
global_results_gbm_all.to_csv("global_results_gbm_all.csv", index=False)
threshold_results_gbm_all.to_csv("threshold_results_gbm_all.csv", index=False)

**Gradient boosting machine mitgated for data deday with importance weights**

In [84]:
from sklearn.ensemble import GradientBoostingRegressor

def run_gbm_evaluation(train_df, test_df, config_label, n_simulations=5, weights=None, gbm_params=None):
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    r2_scores = []
    spearman_corrs = []
    sim_results = {t: {'exclusion_error': [], 'inclusion_error': [], 'targeting_error': []} for t in thresholds}

    for _ in tqdm(range(n_simulations), desc=f"Running {config_label} [GBM]"):
        if train_df.equals(test_df):
            train_split, test_split = train_test_split(train_df, test_size=0.25)
            weight_split = weights.loc[train_split.index] if weights is not None else None
        else:
            train_split, test_split = train_df.copy(), test_df.copy()
            weight_split = weights if weights is not None else None

        y_train = train_split['log_consumption']
        y_test = test_split['log_consumption']

        X_train = train_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = test_split.drop(columns=['Equivalenced consumption', 'log_consumption'])
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        model = GradientBoostingRegressor(random_state=42, **(gbm_params or {}))
        model.fit(X_train, y_train, sample_weight=weight_split)
        y_pred = model.predict(X_test)

        r2_scores.append(r2_score(y_test, y_pred))
        spearman_corrs.append(spearmanr(y_test, y_pred).correlation)

        for q in thresholds:
            ee = exclusion_error_rate(y_test, y_pred, q)
            ie = inclusion_error_rate(y_test, y_pred, q)
            ter = targeting_error_rate(y_test, y_pred, q)
            sim_results[q]['exclusion_error'].append(ee)
            sim_results[q]['inclusion_error'].append(ie)
            sim_results[q]['targeting_error'].append(ter)

    global_summary = pd.DataFrame([{
        'config': config_label,
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'spearman_mean': np.mean(spearman_corrs),
        'spearman_std': np.std(spearman_corrs),
    }])

    threshold_summary = []
    for q in thresholds:
        m = sim_results[q]
        threshold_summary.append({
            'config': config_label,
            'threshold': int(q * 100),
            'ter_mean': np.mean(m['targeting_error']),
            'ter_std': np.std(m['targeting_error']),
            'ee_mean': np.mean(m['exclusion_error']),
            'ee_std': np.std(m['exclusion_error']),
            'ie_mean': np.mean(m['inclusion_error']),
            'ie_std': np.std(m['inclusion_error']),
        })

    return global_summary, pd.DataFrame(threshold_summary)

In [85]:
# Define best GBM parameters per training year
best_gbm_params_2010 = {
    'n_estimators': 200,
    'min_samples_leaf': 1,
    'max_depth': 3,
    'learning_rate': 0.05
}

best_gbm_params_2013 = {
    'n_estimators': 200,
    'min_samples_leaf': 10,
    'max_depth': 3,
    'learning_rate': 0.01
}

# Run GBM with importance weighting

# 2010 → 2013
global_gbm_2013, thresh_gbm_2013 = run_gbm_evaluation(
    harmonized_features2010_encoded,
    harmonized_features2013_encoded,
    '2010→2013 (IW GBM)',
    weights=weights_1030,
    gbm_params=best_gbm_params_2010
)

# 2010 → 2016
global_gbm_2016, thresh_gbm_2016 = run_gbm_evaluation(
    harmonized_features2010_encoded,
    harmonized_features2016_encoded,
    '2010→2016 (IW GBM)',
    weights=weights_1060,
    gbm_params=best_gbm_params_2010
)

# 2013 → 2016
global_gbm_1360, thresh_gbm_1360 = run_gbm_evaluation(
    harmonized_features2013_encoded,
    harmonized_features2016_encoded,
    '2013→2016 (IW GBM)',
    weights=weights_1360,
    gbm_params=best_gbm_params_2013
)

Running 2013→2016 (IW GBM) [GBM]: 100%|██████████| 5/5 [00:07<00:00,  1.52s/it]


In [86]:
global_results_IW_gbm_all = pd.concat([global_gbm_2013, global_gbm_2016, global_gbm_1360], ignore_index=True)
threshold_results_IW_gbm_all = pd.concat([thresh_gbm_2013, thresh_gbm_2016, thresh_gbm_1360], ignore_index=True)

global_results_IW_gbm_all.to_csv("global_results_IW_gbm_all.csv", index=False)
threshold_results_IW_gbm_all.to_csv("threshold_results_IW_gbm_all.csv", index=False)

-----